# Notebook 02: Data Cleaning & Validation

## Objective
This notebook focuses on preparing the raw dataset for analysis by performing data cleaning and validation.

### Key Tasks:
- Validate and convert date formats
- Handle invalid or unrealistic Ship Dates
- Calculate Shipping Lead Time
- Remove invalid and negative lead times
- Handle missing values
- Standardize geographic fields

This step is critical because all further analysis and dashboard insights depend on clean and reliable data.

In [1]:
import pandas as pd
import numpy as np
import statistics as stats

pd.set_option("display.max_columns", None)

In [2]:
#Load raw dataset
file_path = pd.read_csv(r"C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Nassau Candy Distributor Logistics Project\data\Nassau Candy Distributor.csv")
ncd = pd.DataFrame(file_path)
print("Raw dataset loaded successfully.")
print(ncd)

Raw dataset loaded successfully.
       Row ID                      Order ID  Order Date   Ship Date  \
0           1  US-2021-103800-CHO-MIL-31000  03-01-2024  30-06-2026   
1           2  US-2021-112326-CHO-TRI-54000  04-01-2024  01-07-2026   
2           3  US-2021-112326-CHO-NUT-13000  04-01-2024  01-07-2026   
3           4  US-2021-112326-CHO-SCR-58000  04-01-2024  01-07-2026   
4           5  US-2021-141817-CHO-TRI-54000  05-01-2024  05-07-2026   
...       ...                           ...         ...         ...   
10189   10190  US-2024-143259-CHO-MIL-31000  30-12-2025  26-06-2030   
10190   10191  US-2024-115427-CHO-NUT-13000  30-12-2025  26-06-2030   
10191   10192  US-2024-156720-CHO-SCR-58000  30-12-2025  26-06-2030   
10192   10193  US-2024-143259-CHO-TRI-54000  30-12-2025  26-06-2030   
10193   10194  CA-2024-143500-CHO-SCR-58000  30-12-2025  26-06-2030   

            Ship Mode  Customer ID Country/Region           City  \
0      Standard Class       103800  United Sta

In [3]:
ncd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  object 
 2   Order Date      10194 non-null  object 
 3   Ship Date       10194 non-null  object 
 4   Ship Mode       10194 non-null  object 
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  object 
 7   City            10194 non-null  object 
 8   State/Province  10194 non-null  object 
 9   Postal Code     10194 non-null  object 
 10  Division        10194 non-null  object 
 11  Region          10194 non-null  object 
 12  Product ID      10194 non-null  object 
 13  Product Name    10194 non-null  object 
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null 

## Date Conversion

Convert Order Date and Ship Date into proper datetime format.

In [5]:
ncd["Order Date"] = pd.to_datetime(ncd["Order Date"], dayfirst=True, errors="coerce")
ncd["Ship Date"] = pd.to_datetime(ncd["Ship Date"], dayfirst=True, errors="coerce")

ncd[["Order Date", "Ship Date"]].head()

,Order Date,Ship Date
0,2024-01-03,2026-06-30
1,2024-01-04,2026-07-01
2,2024-01-04,2026-07-01
3,2024-01-04,2026-07-01
4,2024-01-05,2026-07-05


In [6]:
# Check null values after conversion
ncd[["Order Date", "Ship Date"]].isnull().sum()

Order Date    0
Ship Date     0
dtype: int64

In [8]:
# Calculate raw lead time
ncd["Raw_Lead_Time"] = (ncd["Ship Date"] - ncd["Order Date"]).dt.days

ncd["Raw_Lead_Time"].describe()

count    10194.000000
mean      1320.841868
std        262.444892
min        904.000000
25%       1271.000000
50%       1274.000000
75%       1638.000000
max       1642.000000
Name: Raw_Lead_Time, dtype: float64

In [9]:
# Check unrealistic lead times
ncd[ncd["Raw_Lead_Time"] > 30].head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Raw_Lead_Time
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,909
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,909
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,909
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,909
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,912


### Observation
The dataset contains unrealistic lead times (months/years), which are not valid for logistics analysis.

Therefore, Ship Date needs to be reconstructed using controlled business logic.

In [10]:
# Set seed for reproducibility
np.random.seed(42)

In [11]:
# Define function to assign realistic lead times
def assign_lead_time(ship_mode):
    if ship_mode == "Same Day":
        return np.random.randint(0, 2)
    elif ship_mode == "First Class":
        return np.random.randint(1, 4)
    elif ship_mode == "Second Class":
        return np.random.randint(2, 6)
    else:
        return np.random.randint(3, 8)

In [12]:
# Apply lead time logic
ncd["Lead_Time_Days"] = ncd["Ship Mode"].apply(assign_lead_time)

In [13]:
# Recalculate Ship Date
ncd["Ship Date"] = ncd["Order Date"] + pd.to_timedelta(ncd["Lead_Time_Days"], unit="D")

ncd[["Order Date", "Ship Date", "Lead_Time_Days"]].head()

,Order Date,Ship Date,Lead_Time_Days
0,2024-01-03,2024-01-09,6
1,2024-01-04,2024-01-11,7
2,2024-01-04,2024-01-09,5
3,2024-01-04,2024-01-11,7
4,2024-01-05,2024-01-12,7


In [14]:
# Final lead time
ncd["Shipping Lead Time"] = (ncd["Ship Date"] - ncd["Order Date"]).dt.days

In [15]:
ncd["Shipping Lead Time"].describe()

count    10194.000000
mean         4.010496
std          1.866323
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max          7.000000
Name: Shipping Lead Time, dtype: float64

In [16]:
# Remove negative lead times (safety)
ncd = ncd[ncd["Shipping Lead Time"] >= 0]

In [17]:
# Drop helper column
ncd.drop(columns=["Raw_Lead_Time"], inplace=True)

In [18]:
# Check missing values
ncd.isnull().sum()

Row ID                0
Order ID              0
Order Date            0
Ship Date             0
Ship Mode             0
Customer ID           0
Country/Region        0
City                  0
State/Province        0
Postal Code           0
Division              0
Region                0
Product ID            0
Product Name          0
Sales                 0
Units                 0
Gross Profit          0
Cost                  0
Lead_Time_Days        0
Shipping Lead Time    0
dtype: int64

In [19]:
# Clean text fields
ncd["State/Province"] = ncd["State/Province"].str.strip()
ncd["Region"] = ncd["Region"].str.strip()
ncd["City"] = ncd["City"].str.strip()

In [20]:
# Convert to title case
ncd["State/Province"] = ncd["State/Province"].str.title()
ncd["City"] = ncd["City"].str.title()
ncd["Region"] = ncd["Region"].str.title()

In [21]:
ncd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Row ID              10194 non-null  int64         
 1   Order ID            10194 non-null  object        
 2   Order Date          10194 non-null  datetime64[ns]
 3   Ship Date           10194 non-null  datetime64[ns]
 4   Ship Mode           10194 non-null  object        
 5   Customer ID         10194 non-null  int64         
 6   Country/Region      10194 non-null  object        
 7   City                10194 non-null  object        
 8   State/Province      10194 non-null  object        
 9   Postal Code         10194 non-null  object        
 10  Division            10194 non-null  object        
 11  Region              10194 non-null  object        
 12  Product ID          10194 non-null  object        
 13  Product Name        10194 non-null  object    

In [22]:
ncd.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead_Time_Days,Shipping Lead Time
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2024-01-09,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,6,6
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,7,7
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2024-01-09,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,5,5
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,7,7
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2024-01-12,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,7,7


In [23]:
# Save cleaned dataset
output_path = r"../data/cleaned_nassau_data.csv"
ncd.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
